# L.11 — Data Preprocessing Notebook

This notebook documents the preprocessing steps used in the UX metrics analysis: loading raw files, cleaning, handling nulls, parsing timestamps, deduplication strategies, and exporting cleaned datasets. Copy/paste the code cells below into a Python REPL or Jupyter notebook to run.

In [ ]:
# Imports and paths
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('/Users/gvnd/Documents/College/Skripsian/data')
TOT_PATH = DATA_DIR / 'tot_results.csv'
QUESTIONNAIRE_PATH = DATA_DIR / 'Usability Testing Aplikasi Mindscape (Responses) (1).xlsx'
OUTPUT_DIR = DATA_DIR / 'processed'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Data dir:', DATA_DIR)

In [ ]:
# Quick preview of raw files (safe, read-only)
tot = pd.read_csv(TOT_PATH)
quest = pd.read_excel(QUESTIONNAIRE_PATH)
print('tot shape:', tot.shape)
print('quest shape:', quest.shape)
print('
TOT columns:', list(tot.columns))
print('
Questionnaire columns (first 20):', list(quest.columns[:20]))

## 1 — Normalize participant keys and basic types
Create a stable participant key and cast numeric fields. This prevents mismatched joins and hidden NaNs.

In [ ]:
# Create a clean participant key used for joins
tot['nama_tester_clean'] = tot['Nama Tester'].astype(str).str.strip().str.lower()
quest['nama_tester_clean'] = quest['Nama Tester'].astype(str).str.strip().str.lower()

# coerced numeric fields and derived columns
tot['tot_ms'] = pd.to_numeric(tot.get('tot_ms', None), errors='coerce')
tot['tot_sec'] = tot['tot_ms'] / 1000.0

# parse timestamps safely (coerce invalids to NaT)
tot['logged_at'] = pd.to_datetime(tot.get('logged_at', None), errors='coerce', utc=False)

# basic checks
print('tot tot_ms nulls:', tot['tot_ms'].isna().sum())
print('tot logged_at nulls:', tot['logged_at'].isna().sum())

## 2 — Questionnaire composites and item casting
Cast item responses to numeric and compute composite scores (ERI/PEI). Keep original items for reproducibility.

In [ ]:
# Identify ERI/PEI item columns by index range used in thesis pipeline
eri_cols = list(quest.columns[4:12])
pei_cols = list(quest.columns[12:20])
# Coerce to numeric, preserving NaNs
for c in eri_cols + pei_cols:
    quest[c] = pd.to_numeric(quest[c], errors='coerce')

quest['ERI_Composite'] = quest[eri_cols].mean(axis=1)
quest['PEI_Composite'] = quest[pei_cols].mean(axis=1)

print('ERI cols sample:', eri_cols)
print('PEI cols sample:', pei_cols)
print('Composite nulls:', quest['ERI_Composite'].isna().sum(), quest['PEI_Composite'].isna().sum())

## 3 — Deduplication and ToT resolution strategies
The notebook provides 3 deterministic strategies used in the analysis: mean, latest, earliest. Keep all strategies for auditability.

In [ ]:
# Duplicate handling
stable_keys = ['nama_tester_clean', 'ui_condition']
dup_counts = (
    tot.groupby(stable_keys, dropna=False)
       .size()
       .rename('n_rows')
       .reset_index()
)
dup_counts['is_duplicate'] = dup_counts['n_rows'] > 1
print('Total key groups:', len(dup_counts))
print('Groups with duplicates:', int(dup_counts['is_duplicate'].sum()))

# Latest, earliest, and mean strategies (keep outputs)
latest_tot = (
    tot.sort_values('logged_at')
       .groupby(stable_keys, as_index=False)
       .tail(1)
       .copy()
)
earliest_tot = (
    tot.sort_values('logged_at')
       .groupby(stable_keys, as_index=False)
       .head(1)
       .copy()
)
mean_tot = (
    tot.groupby(stable_keys, as_index=False)
       .agg(tot_sec=('tot_sec', 'mean'))
)
# Save audit copies
latest_tot.to_csv(OUTPUT_DIR / 'latest_tot.csv', index=False)
earliest_tot.to_csv(OUTPUT_DIR / 'earliest_tot.csv', index=False)
mean_tot.to_csv(OUTPUT_DIR / 'mean_tot.csv', index=False)
print('Saved deduplicated variants to', OUTPUT_DIR)

## 4 — Merge questionnaire with resolved ToT and export paired dataset
Pivot ToT to per-participant per-condition and merge with questionnaire. Export `paired` for statistical analysis.

In [ ]:
# Use earliest strategy by default for reproducibility
tot_for_thesis = earliest_tot[['nama_tester_clean', 'ui_condition', 'tot_sec']]
pivot = (
    tot_for_thesis.pivot_table(index='nama_tester_clean', columns='ui_condition', values='tot_sec', aggfunc='mean')
    .reset_index()
    .rename(columns={'standard_ui': 'ToT_Standard', 'rush_hour_ui': 'ToT_RushHour'})
)
q_keep = quest[['nama_tester_clean', 'Nama Tester', 'Group', 'ERI_Composite', 'PEI_Composite'] + eri_cols + pei_cols]
merged = pd.merge(q_keep, pivot, on='nama_tester_clean', how='inner')
paired = merged.dropna(subset=['ToT_Standard', 'ToT_RushHour']).copy()
paired['ToT_Savings'] = paired['ToT_Standard'] - paired['ToT_RushHour']

# Export cleaned datasets used for analysis
paired.to_csv(OUTPUT_DIR / 'paired_for_analysis.csv', index=False)
quest.to_csv(OUTPUT_DIR / 'questionnaire_processed.csv', index=False)
tot.to_csv(OUTPUT_DIR / 'tot_processed.csv', index=False)
print('Exported paired and processed files to', OUTPUT_DIR)

## 5 — Minimal validation prints (console logs)
These prints prove the pipeline results (Cronbach, counts, and null checks) are reproducible in the analysis notebook.

In [ ]:
# Quick reproducibility checks used by the analysis pipeline
print('Paired rows:', len(paired))
print('Paired ToT_Savings mean, sd:', paired['ToT_Savings'].mean(), paired['ToT_Savings'].std(ddof=1))
print('Questionnaire composites NA counts (ERI, PEI):', quest['ERI_Composite'].isna().sum(), quest['PEI_Composite'].isna().sum())

# Show head of paired to confirm shape and columns
print(paired.head().to_string(index=False))

---
**Notes & provenance**:
- Keep all intermediate CSVs in `data/processed/` for audit.
- Do not drop raw columns unless explicitly documented.
- If you change the `TOT_STRATEGY`, re-run the full analysis to update tables and figures.